In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

missing_values_path = OUT_DIR / "missing_values_imputed.csv"

# Let pandas detect the delimiter
data = pd.read_csv(missing_values_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data

In [ ]:
data.columns

# Feature Derivation

In [ ]:
# -------------------------------------------------------------------------
# FEATURE DERIVATION PLAN (UPDATED)
# -------------------------------------------------------------------------
# This section outlines the 10 new features derived from the dataset.
#
# For each feature, we describe:
#   • What it represents and why it is relevant
#   • The columns used
#   • The calculation or logic applied
# -------------------------------------------------------------------------


# =========================================================================
# 1. change_in_distance
# -------------------------------------------------------------------------
# Meaning:
#   Change (in km) between the student’s previous address/school and their
#   current home address relative to the main campus (3584_CS).
#   Captures mobility behaviour without enforcing thresholds.
#
# Calculation:
#   change_in_distance = distance_current − distance_previous
#
# Columns used:
#   - sdo1_postal_distance_distance_to_3584_CS
#   - sdo1_prev_distance_distance_to_3584_CS
# =========================================================================


# =========================================================================
# 2. age_category
# -------------------------------------------------------------------------
# Meaning:
#   Groups students into meaningful age segments at the start of their degree
#   based on observed age distribution. Helps capture non-linear effects in age.
#
# Categories:
#   - <17              → 'early_starters'
#   - 17–18            → 'common_student_age'
#   - 19–21            → 'late_starters'
#   - >21              → 'career_changers'
#
# Columns used:
#   - sdo1_characteristics_age_start_study
# =========================================================================


# =========================================================================
# 3. time_first_orient_to_start_weeks
# -------------------------------------------------------------------------
# Meaning:
#   Weeks between the first recorded orientation event and the degree start.
#   Indicates how early the student engaged with the university.
#
# Calculation:
#   (degree_start_date − first_orientation_date) in weeks
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo2_orientation_First_Event_Date
# =========================================================================


# =========================================================================
# 4. time_last_orient_to_start_weeks
# -------------------------------------------------------------------------
# Meaning:
#   Weeks between the last orientation event and the degree start.
#   Smaller values represent engagement close to the start of studies.
#
# Calculation:
#   (degree_start_date − last_orientation_date) in weeks
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo2_orientation_Last_Event_Date
# =========================================================================


# =========================================================================
# 5. gap_skc_to_start_weeks
# -------------------------------------------------------------------------
# Meaning:
#   Weeks between the SKC meeting and the degree start date.
#   Can be positive or negative because SKC may occur before or after
#   the official start (e.g., introduction weeks, first-year follow-up SKC).
#
# Calculation:
#   (degree_start_date − skc_date) in weeks
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo2_skc_SKC_DATUM
# =========================================================================


# =========================================================================
# 6. time_enroll_to_start_weeks
# -------------------------------------------------------------------------
# Meaning:
#   Weeks between the formal enrolment date and the degree start.
#   Reflects administrative timing or the student’s enrolment behaviour.
#
# Calculation:
#   (degree_start_date − enrollment_date) in weeks
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo5_degree_enrollment_date
# =========================================================================


# =========================================================================
# 7. gap_prev_exam_to_start_weeks
# -------------------------------------------------------------------------
# Meaning:
#   Weeks between the student’s final exam from their prior education
#   and the start of their degree. Useful for identifying gap years,
#   direct transitions, or delayed starts.
#
# Calculation:
#   (degree_start_date − previous_exam_date) in weeks
#
# Columns used:
#   - sdo5_degree_degree_start_date
#   - sdo1_previous_Final_Exam_Date
# =========================================================================


# =========================================================================
# 8. has_open_day
# -------------------------------------------------------------------------
# Meaning:
#   Indicates whether the student attended an Open Day–related activity.
#
# Calculation:
#   1 if Event_Types_Attended ∈ ['Open_Day_Related', 'Multiple_Types'] else 0
#
# Columns used:
#   - sdo2_orientation_Event_Types_Attended
# =========================================================================


# =========================================================================
# 9. has_proefstuderen
# -------------------------------------------------------------------------
# Meaning:
#   Indicates whether the student participated in a Trial/Experience Day
#   (Proefstuderen).
#
# Calculation:
#   1 if Event_Types_Attended ∈ ['Trial_or_Experience_Day', 'Multiple_Types'] else 0
#
# Columns used:
#   - sdo2_orientation_Event_Types_Attended
# =========================================================================


# =========================================================================
# 10. has_campus_tour
# -------------------------------------------------------------------------
# Meaning:
#   Indicates whether the student attended a campus tour event.
#
# Calculation:
#   1 if Event_Types_Attended ∈ ['Campus_Tour', 'Multiple_Types'] else 0
#
# Columns used:
#   - sdo2_orientation_Event_Types_Attended
# =========================================================================

# NOTE:
# The original variable `has_orientation` from the dataset is preserved as-is
# and is not re-derived here.


In [ ]:
data[['sdo2_skc_SKC_DATUM', 'sdo5_degree_degree_start_date']].head(10)

In [ ]:
# ------------------------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------------------------
# Derive 10 new features from the current dataset. All time deltas are in WEEKS.
# Uses the existing start date column: `sdo5_degree_degree_start_date`.
#
# New columns created:
#   1) change_in_distance                         (float, km)
#   2) age_category                               (category: 'early_starters',
#                                                  'common_student_age',
#                                                  'late_starters',
#                                                  'career_changers')
#   3) time_first_orient_to_start_weeks           (float, weeks)
#   4) time_last_orient_to_start_weeks            (float, weeks)
#   5) gap_skc_to_start_weeks                     (float, weeks)
#   6) time_enroll_to_start_weeks                 (float, weeks)
#   7) gap_prev_exam_to_start_weeks               (float, weeks)
#   8) has_open_day                               (Int64: {0,1})
#   9) has_proefstuderen                          (Int64: {0,1})
#  10) has_campus_tour                            (Int64: {0,1})
#
# Notes:
# - Keep original features: sdo6_results_Percentage_Credits_A and _B (no diff derived).
# - `has_orientation` from the original data is kept as-is (no override).
# - Robust to missing columns/values (returns NaN/NaT where unavailable).
# - `_weeks_between` preserves sign (later - earlier), expressed in weeks.
# ------------------------------------------------------------------------------


# ---------- Safe getters ----------
def _get_dt(df: pd.DataFrame, col: str) -> pd.Series:
    s = df.get(col)
    if s is None:
        return pd.Series(pd.NaT, index=df.index)
    return pd.to_datetime(s, errors='coerce')

def _get_num(df: pd.DataFrame, col: str) -> pd.Series:
    s = df.get(col)
    if s is None:
        return pd.Series(np.nan, index=df.index)
    return pd.to_numeric(s, errors='coerce')

def _weeks_between(later: pd.Series, earlier: pd.Series) -> pd.Series:
    """
    Compute (later - earlier) in weeks as float.
    Preserves sign. Returns NaN where either side is NaT.
    """
    dt = pd.to_datetime(later, errors='coerce') - pd.to_datetime(earlier, errors='coerce')
    return (dt / pd.Timedelta(weeks=1)).astype(float)

# ---------- Main derivation ----------
def derive_features(df, age_bins=None, age_labels=None, rounding=2) -> pd.DataFrame:
    """
    Derive the 10 requested features. Returns a new DataFrame (copy).
    Robust to missing columns: missing dates → NaT; missing numerics → NaN.
    """
    df_out = df.copy()

    # Numerics (kept as originals, no difference feature derived)
    perc_A    = _get_num(df_out, 'sdo6_results_Percentage_Credits_A')
    perc_B    = _get_num(df_out, 'sdo6_results_Percentage_Credits_B')
    dist_curr = _get_num(df_out, 'sdo1_postal_distance_distance_to_3584_CS')
    dist_prev = _get_num(df_out, 'sdo1_prev_distance_distance_to_3584_CS')
    age_start = _get_num(df_out, 'sdo1_characteristics_age_start_study')

    # Dates
    deg_start       = _get_dt(df_out, 'sdo5_degree_degree_start_date')
    first_orient    = _get_dt(df_out, 'sdo2_orientation_First_Event_Date')
    last_orient     = _get_dt(df_out, 'sdo2_orientation_Last_Event_Date')
    skc_date        = _get_dt(df_out, 'sdo2_skc_SKC_DATUM')
    enroll_date     = _get_dt(df_out, 'sdo5_degree_enrollment_date')
    prev_exam_date  = _get_dt(df_out, 'sdo1_previous_Final_Exam_Date')  # NEW

    # 1) Distance change (only distance-based feature kept)
    df_out['change_in_distance'] = (dist_curr - dist_prev).round(rounding)

    # 2) Age category bins
    # New groups based on your distribution check:
    #   <17          → 'early_starters'
    #   17–18        → 'common_student_age'
    #   19–21        → 'late_starters'
    #   >21          → 'career_changers'
    if age_bins is None:
        age_bins = [-np.inf, 17, 19, 22, np.inf]
    if age_labels is None:
        age_labels = [
            'early_starters',
            'common_student_age',
            'late_starters',
            'career_changers',
        ]
    df_out['age_category'] = pd.cut(
        age_start,
        bins=age_bins,
        labels=age_labels,
        right=False,           # [a, b) so we get <17, 17–18, 19–21, >21
        include_lowest=True
    )

    # 3–7) Week-based time deltas (signed)
    df_out['time_first_orient_to_start_weeks'] = _weeks_between(deg_start, first_orient).round(rounding)
    df_out['time_last_orient_to_start_weeks']  = _weeks_between(deg_start, last_orient).round(rounding)
    df_out['gap_skc_to_start_weeks']           = _weeks_between(deg_start, skc_date).round(rounding)
    df_out['time_enroll_to_start_weeks']       = _weeks_between(deg_start, enroll_date).round(rounding)
    df_out['gap_prev_exam_to_start_weeks']     = _weeks_between(deg_start, prev_exam_date).round(rounding)

    # 8–10) Event-type flags from attended category
    evt = df_out.get('sdo2_orientation_Event_Types_Attended',
                     pd.Series(index=df_out.index, dtype='object'))
    evt_norm = evt.astype('string').str.strip().fillna('')

    df_out['has_open_day']      = evt_norm.isin(['Open_Day_Related', 'Multiple_Types']).astype('Int64')
    df_out['has_proefstuderen'] = evt_norm.isin(['Trial_or_Experience_Day', 'Multiple_Types']).astype('Int64')
    df_out['has_campus_tour']   = evt_norm.isin(['Campus_Tour', 'Multiple_Types']).astype('Int64')

    # --- Keep original has_orientation (no changes) ---
    if 'has_orientation' in df_out.columns:
        df_out['has_orientation'] = df_out['has_orientation'].astype('Int64')

    return df_out

# -------------------------------
# RUN & QUICK INSPECTION
# -------------------------------
data = derive_features(data, rounding=2)

# Preview key outputs (includes NEW gap_prev_exam_to_start_weeks)
display(
    data.filter(regex="^sdo5_degree_degree_start_date$|change_in_distance|age_category|_weeks$|^has_").head(15)
)

print("\nSummary of time deltas (in weeks):")
print(data.filter(regex="_weeks$").describe().T)


# Bonus Part: DERIVING NEW FEATURES WITH NEW CATEGOGIES TO ESCAPE NEGATIVE VALUES


In [ ]:
# ------------------------------------------------------------------------------
# INTERPRETING NEGATIVE VALUES AND DERIVING NEW CATEGORICAL FEATURES
# ------------------------------------------------------------------------------
# In the previous step, we derived several numerical features expressed as
# differences in weeks or continuous values. Some of these features can take negative values.
#
# Below we interpret each and outline the corresponding categorical features that
# can be derived to make the data more interpretable.
# ------------------------------------------------------------------------------

# =========================================================================
# 1. gap_skc_to_start_weeks → SKC timing interpretation
# -------------------------------------------------------------------------
# • Positive values: SKC happened BEFORE the degree start date (expected).
# • Negative values: SKC happened AFTER the degree start date (unusual or delayed).
#
# New categorical feature:
#   skc_timing_category:
#       - 'skc_happened_earlier'  → gap_skc_to_start_weeks > 0
#       - 'skc_happened_late'     → gap_skc_to_start_weeks < 0
#       - 'no_gap'                → gap_skc_to_start_weeks == 0
# =========================================================================


# =========================================================================
# 2. performance_diff_blocks → Academic trend interpretation
# -------------------------------------------------------------------------
# • Positive values: Performance improved from Block A to Block B.
# • Negative values: Performance declined between the two blocks.
# • Zero values: Performance remained constant.
#
# New categorical feature:
#   performance_trend_category:
#       - 'improved'   → performance_diff_blocks > 0
#       - 'decreased'   → performance_diff_blocks < 0
#       - 'no_change'  → performance_diff_blocks == 0
# =========================================================================


In [ ]:
# ------------------------------------------------------------------------------
# CREATE CATEGORICAL VERSIONS OF TWO DIRECTIONAL FEATURES
# ------------------------------------------------------------------------------
# This step starts by CALCULATING `performance_diff_blocks` directly from:
#   - sdo6_results_Percentage_Credits_A
#   - sdo6_results_Percentage_Credits_B
#
# Derived categorical features:
#   - skc_timing_category
#   - performance_trend_category
#
# Robustness:
# - Validates required numeric columns exist.
# - Preserves missing values (<NA>).
# - Uses vectorized np.select for clarity and speed.
# ------------------------------------------------------------------------------

def _require_cols(df, cols):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required column(s): {missing}. Please check your inputs.")

# ---------------------------
# 0) Calculate performance_diff_blocks instantly
# ---------------------------
_required_perf = [
    'sdo6_results_Percentage_Credits_A',
    'sdo6_results_Percentage_Credits_B'
]
_require_cols(data, _required_perf)

data['performance_diff_blocks'] = (
    pd.to_numeric(data['sdo6_results_Percentage_Credits_B'], errors='coerce')
    - pd.to_numeric(data['sdo6_results_Percentage_Credits_A'], errors='coerce')
)

# ---------------------------
# 1) SKC timing category
# ---------------------------
_require_cols(data, ['gap_skc_to_start_weeks'])

skc = data['gap_skc_to_start_weeks']

data['skc_timing_category'] = pd.Series(
    np.select(
        condlist=[skc > 0, skc < 0, skc == 0],
        choicelist=['skc_happened_earlier', 'skc_happened_late', 'no_gap'],
        default=pd.NA
    ),
    index=data.index,
    dtype='object'
).astype('category')

# ---------------------------
# 2) Performance trend category
# ---------------------------
perf = data['performance_diff_blocks']

data['performance_trend_category'] = pd.Series(
    np.select(
        condlist=[perf > 0, perf < 0, perf == 0],
        choicelist=['improved', 'decreased', 'no_change'],
        default=pd.NA
    ),
    index=data.index,
    dtype='object'
).astype('category')

# ---------------------------
# Optional: sanity checks
# ---------------------------
print("skc_timing_category:\n", data['skc_timing_category'].value_counts(dropna=False))
print("\nperformance_trend_category:\n", data['performance_trend_category'].value_counts(dropna=False))


In [ ]:
data[['sdo2_skc_SKC_DATUM', 'sdo5_degree_degree_start_date']].head(10)

In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False).head(20)
print(nan_sums)

#  Concern Over Duplicates after feature derivation

We need to brainstorm on what to remove/leave at the end. 
The below code attempts raise the concerns

In [ ]:

# ---------------------------
# Check Possible duplicate columns
# ---------------------------

# 1) Check if the two IDs are the same
same_id = (data['SINH_ID'].astype(str).str.strip()
           == data['sdo2_orientation_STUDENTNUMMER'].astype(str).str.strip())
print("Student ID equality rate:", same_id.mean())

# 2) Check if previous-school postcode columns match
if 'sdo1_prev_distance_postcode' in data and 'sdo1_previous_Previous_School_Postal_Code' in data:
    same_prev_pc = (data['sdo1_prev_distance_postcode'].astype(str).str.strip()
                    == data['sdo1_previous_Previous_School_Postal_Code'].astype(str).str.strip())
    print("Prev-school postcode equality rate:", same_prev_pc.mean())

# 3) Orientation presence redundancy
if 'has_orientation' in data and 'has_any_orientation' in data:
    print("has_orientation vs has_any_orientation agreement:",
          (data['has_orientation'].astype('Int64') == data['has_any_orientation'].astype('Int64')).mean())

# 4) SKC presence redundancy
if 'has_skc' in data and 'sdo2_skc_SKC_DATUM_missing_flag' in data:
    # If missing flag means "date missing", invert to compare to has_skc.
    inv_missing = 1 - data['sdo2_skc_SKC_DATUM_missing_flag'].astype('Int64')
    print("has_skc vs not-missing SKC date agreement:",
          (data['has_skc'].astype('Int64') == inv_missing).mean())


In [ ]:
# =============================================================================
# PURPOSE
# -----------------------------------------------------------------------------
# Remove the redundant SKC missing-date flag.
#
# Reason:
#   • The small mismatch (~0.3%) between has_skc and the SKC date flag is a
#     minor data-quality quirk, not a structural issue.
#   • has_skc is the authoritative indicator of SKC presence.
#   • sdo2_skc_SKC_DATUM_missing_flag adds noise and should be dropped.
# =============================================================================

col_to_drop = "sdo2_skc_SKC_DATUM_missing_flag"

if col_to_drop in data.columns:
    data = data.drop(columns=[col_to_drop])
    print(f"Dropped column: {col_to_drop}")
else:
    print(f"Column not found: {col_to_drop}")

In [ ]:
# =============================================================================
# PURPOSE
# -----------------------------------------------------------------------------
# Drop columns confirmed as redundant or no longer needed for modeling.
#
# Columns to drop:
#   • sdo5_degree_degree_start_date
#        - Redundant; only needed for earlier exclusion logic (handled already).
#
#   • sdo1_characteristics_age_start_study
#        - Dropped because age_category replaces it.
#
#   • sdo1_prev_distance_distance_to_3812_PA
#   • sdo1_postal_distance_distance_to_3812_PA
#        - Distances to Amersfoort buildings; irrelevant for this dataset.
#
#   • sdo2_orientation_STUDENTNUMMER
#        - Redundant; SINH_ID is the canonical student identifier.
#
#   • performance_diff_blocks
#        - Not part of the 10 feature derivations; removed to prevent confusion.
#
# IMPORTANT:
#   • Do NOT drop has_orientation.
# =============================================================================

cols_to_drop = [
    "sdo5_degree_degree_start_date",
    "sdo1_characteristics_age_start_study",
    "sdo1_prev_distance_distance_to_3812_PA",
    "sdo1_postal_distance_distance_to_3812_PA",
    "sdo2_orientation_STUDENTNUMMER",
    "performance_diff_blocks"   # <-- newly added
]

existing_to_drop = [c for c in cols_to_drop if c in data.columns]

print("Dropping columns:", existing_to_drop)
before = data.shape
data = data.drop(columns=existing_to_drop, errors="ignore")
after = data.shape

print(f"Shape before: {before} → after: {after}")


In [ ]:
# ------------------------------------------------------------------------------
# INSPECT: COLUMNS WITH '_missing_flag'
# ------------------------------------------------------------------------------
# This cell identifies all columns that serve as detailed missingness indicators.
# These columns often duplicate the simpler 'has_*' flags and can be removed safely.
# ------------------------------------------------------------------------------

missing_flag_cols = [c for c in data.columns if c.endswith('_missing_flag')]

print(f"Total '_missing_flag' columns found: {len(missing_flag_cols)}\n")

# Optionally group them by their base prefix (everything before the last underscore)
from collections import defaultdict
grouped_flags = defaultdict(list)
for col in missing_flag_cols:
    base = col.split('_missing_flag')[0]
    group_root = base.split('_')[0]  # rough grouping by sdo1/sdo2/sdo6 etc.
    grouped_flags[group_root].append(col)

for group, cols in grouped_flags.items():
    print(f"--- {group.upper()} ({len(cols)} columns) ---")
    for c in cols:
        print("  ", c)
    print()

# If you prefer a simple list instead of grouped view:
# print(missing_flag_cols)

In [ ]:
# =============================================================================
# PURPOSE
# -----------------------------------------------------------------------------
# Drop all *_missing_flag columns that correspond to variables
# already removed in the previous cleanup step.
#
# This ensures no orphaned missingness indicators remain after deleting:
#   • sdo5_degree_degree_start_date
#   • sdo1_characteristics_age_start_study
#   • sdo1_prev_distance_distance_to_3812_PA
#   • sdo1_postal_distance_distance_to_3812_PA
#   • sdo2_orientation_STUDENTNUMMER
#
# (We KEEP has_orientation — do not delete its flag.)
# =============================================================================

# Columns deleted earlier
deleted_cols = [
    "sdo5_degree_degree_start_date",
    "sdo1_characteristics_age_start_study",
    "sdo1_prev_distance_distance_to_3812_PA",
    "sdo1_postal_distance_distance_to_3812_PA",
    "sdo2_orientation_STUDENTNUMMER",
]

# Build list of corresponding flags to remove
flags_to_drop = [
    f"{col}_missing_flag"
    for col in deleted_cols
    if f"{col}_missing_flag" in data.columns
]

print("Missing-flag columns to drop:", flags_to_drop)

before = data.shape
data = data.drop(columns=flags_to_drop, errors="ignore")
after = data.shape

print(f"\nShape before: {before} → after: {after}")


# Check again from the remaining columns, is there a possibility of redundant columns?

what is our reason for keeping them? 

Save final data frame after this check, ready for one-hot-encoding

In [ ]:
data.columns

In [ ]:
# ------------------------------------------------------------------------------
#Opinion on the remaining columns
# ------------------------------------------------------------------------------
# Assess whether any remaining columns are true duplicates or redundant.
#
# Summary of findings:
#   • No actual duplicate columns are present in the current dataset.
#
#   • Several groups encode the same underlying concept in different but
#     complementary ways (numeric, categorical, binary), which is intentional:
#
#       – Orientation:
#           • Numeric counts (intensity of participation)
#           • Missing flags (structural absence)
#           • has_orientation (simple binary indicator)
#         These are not duplicates; each adds different modeling value.
#
#       – SKC:
#           • Raw SKC date
#           • gap_skc_to_start_weeks (numeric interval)
#           • skc_timing_category (categorical version)
#           • has_skc (binary presence indicator)
#         These represent different perspectives of the same event.
#
#       – Performance:
#           • Numeric academic metrics (percentage, potential, grades)
#           • performance_trend_category (categorical interpretation)
#         These are intentionally complementary, not redundant.
#
#       – Distances:
#           • Previous vs current home distances are genuinely different features.
#
#   • Missing flags correspond to numeric variables and are kept for interpretability.
#
# Conclusion:
#   The dataset contains no problematic duplicates. All remaining columns serve
#   distinct or intentionally complementary roles and should be retained.
# ------------------------------------------------------------------------------


In [ ]:
# Save the cleaned dataset
output_path = OUT_DIR / "features_derived.csv"
data.to_csv(output_path, index=False)

# Print summary of final dataset
print("✅ Feature derivation completed successfully!")
print(f"📁 Saved to: {output_path}")
print(f"\n📊 Final Dataset Statistics:")
print(f"   • Total rows: {len(data):,}")
print(f"   • Total columns: {len(data.columns)}")
print(f"   • Memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\n🎓 Ready for feature normalization!")

# Ignore the two cells below, I just put them there out of curiosity

It was just a check to see what numeric or categorical columns we have

In [ ]:
# ------------------------------------------------------------------------------
# CLEANUP VERSION 1: KEEP NUMERIC FEATURES (drop categorical/binary duplicates)
# ------------------------------------------------------------------------------
# Creates `data_numeric_pref` — a version of the dataset prioritizing numeric
# representations for modeling. Categorical or binary duplicates are dropped.
# ------------------------------------------------------------------------------

data_numeric_pref = data.copy()

# Only include columns that exist in the dataset
drop_for_numeric_pref = [
    'performance_trend_category',     # exists
    'skc_timing_category',            # exists
    'has_open_day',                   # exists
    'has_proefstuderen',              # exists
    'has_campus_tour',                # exists
    'age_category'                    # exists
    # Removed: distance_change_category (does NOT exist)
    # Removed: moved_for_study (does NOT exist)
    # Removed: has_any_orientation (does NOT exist)
]

# Drop only those that actually exist
cols_to_drop = [c for c in drop_for_numeric_pref if c in data_numeric_pref.columns]

data_numeric_pref = data_numeric_pref.drop(columns=cols_to_drop, errors='ignore')

print(f"Numeric-preferred dataset shape: {data_numeric_pref.shape}")
print("Dropped categorical/binary duplicates:", cols_to_drop)

In [ ]:
# ------------------------------------------------------------------------------
# CLEANUP VERSION 2: KEEP CATEGORICAL FEATURES (drop numeric duplicates)
# ------------------------------------------------------------------------------
# Creates `data_categorical_pref` — a version of the dataset prioritizing
# categorical or interpretable features for easier visualization and encoding.
# Numeric duplicates are dropped.
# ------------------------------------------------------------------------------

data_categorical_pref = data.copy()

drop_for_categorical_pref = [
    # Drop numeric versions of derived relationships
    'change_in_distance',             # numeric version of (not-kept) distance category
    'gap_skc_to_start_weeks',         # numeric version of skc_timing_category
    'time_first_orient_to_start_weeks',
    'time_last_orient_to_start_weeks',
    'time_enroll_to_start_weeks',     # numeric timing; categorical views preferred here
    # 'sdo1_characteristics_age_start_study'  # removed: column no longer exists
]

cols_to_drop_cat = [c for c in drop_for_categorical_pref if c in data_categorical_pref.columns]

data_categorical_pref = data_categorical_pref.drop(
    columns=cols_to_drop_cat,
    errors='ignore'
)

print(f"Categorical-preferred dataset shape: {data_categorical_pref.shape}")
print("Dropped numeric duplicates:", cols_to_drop_cat)